# 02: 文档加载 - 处理真实世界的数据

## 学习目标

在RAG（检索增强生成）系统中，**文档加载（Document Loading）** 是整个pipeline的第一步，也是最关键的基础环节。无论你的检索算法多么精妙、生成模型多么强大，如果文档加载阶段丢失了信息或引入了噪音，后续所有步骤都建立在错误的基础上。

### 本课涵盖的内容：

1. **统一文档接口** —— 定义 `Document` 数据类和 `DocumentLoader` 抽象基类，确保所有加载器遵循相同的契约
2. **PDF加载** —— 使用 PyPDF2 和 pdfplumber 两种策略，对比提取质量差异
3. **Word文档加载** —— 处理 .docx 中的段落、表格、页眉页脚
4. **Markdown加载** —— 将格式化标记转换为纯文本，保留语义结构
5. **HTML加载** —— 使用 BeautifulSoup 清洗网页内容，去除脚本和样式
6. **OCR加载** —— 使用 Tesseract + PIL 从扫描文档/图片中提取文字
7. **工厂模式** —— 根据文件扩展名自动选择合适的加载器
8. **批量加载** —— 递归遍历目录，加载所有支持的文件类型

### 错误处理哲学

真实世界的文档充满了"意外"：损坏的文件、加密的PDF、空文档、编码问题、缺失的依赖... 本课采用 **"优雅降级，永不崩溃"** 的原则：

- 每个加载器都包含完整的 try/except 处理
- 单个文件失败不影响批量加载的其他文件
- 错误信息记录在 Document.metadata 中，便于调试
- 空文档返回空列表而非 None，保持接口一致性

让我们开始构建一个健壮的文档加载系统。

## 1. 环境准备

以下是本课所需的依赖库。如果你尚未安装，请取消注释对应的 pip install 行并运行。

In [ ]:
# === 环境准备：安装依赖 ===
# 取消注释以下行来安装所需库
# !pip install PyPDF2
# !pip install pdfplumber
# !pip install python-docx
# !pip install markdown
# !pip install beautifulsoup4
# !pip install pytesseract
# !pip install Pillow
# !pip install fpdf2

# === 标准库导入 ===
import os
import sys
import json
import tempfile
import shutil
from pathlib import Path
from typing import Optional, Callable
from dataclasses import dataclass, field
from abc import ABC, abstractmethod

# === 第三方库导入 ===
# PDF
try:
    from PyPDF2 import PdfReader
    PYPDF2_AVAILABLE = True
except ImportError:
    PYPDF2_AVAILABLE = False
    print("[WARNING] PyPDF2 未安装。运行: pip install PyPDF2")

try:
    import pdfplumber
    PDFPLUMBER_AVAILABLE = True
except ImportError:
    PDFPLUMBER_AVAILABLE = False
    print("[WARNING] pdfplumber 未安装。运行: pip install pdfplumber")

# Word
try:
    from docx import Document as DocxDocument
    DOCX_AVAILABLE = True
except ImportError:
    DOCX_AVAILABLE = False
    print("[WARNING] python-docx 未安装。运行: pip install python-docx")

# Markdown
try:
    import markdown
    from markdown.extensions.extra import ExtraExtension
    MARKDOWN_AVAILABLE = True
except ImportError:
    MARKDOWN_AVAILABLE = False
    print("[WARNING] markdown 未安装。运行: pip install markdown")

# HTML
try:
    from bs4 import BeautifulSoup
    BS4_AVAILABLE = True
except ImportError:
    BS4_AVAILABLE = False
    print("[WARNING] beautifulsoup4 未安装。运行: pip install beautifulsoup4")

# OCR
try:
    import pytesseract
    from PIL import Image, ImageDraw, ImageFont
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False
    print("[WARNING] pytesseract 未安装。运行: pip install pytesseract")

# 用于创建测试PDF
try:
    from fpdf import FPDF
    FPDF_AVAILABLE = True
except ImportError:
    FPDF_AVAILABLE = False
    print("[INFO] fpdf2 未安装。将使用备用方法创建测试PDF。运行: pip install fpdf2")

print("\n[OK] 环境检查完成。")

## 2. 统一文档接口

在构建多格式加载系统之前，我们首先定义统一的数据模型和抽象接口。

### 设计原则

- **Document** —— 最小化的数据载体：`text`（提取的文本内容） + `metadata`（来源、页码、错误信息等）
- **DocumentLoader** —— 抽象基类，定义 `load(file_path) -> list[Document]` 契约

所有加载器返回 **列表** 而非单个对象，因为一个文件可能包含多个逻辑文档（例如PDF的每一页可以是一个Document）。这种设计为后续的文本分块（chunking）和索引建立提供了灵活性。

In [ ]:
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from typing import Any


@dataclass
class Document:
    """
    统一的文档数据模型。
    
    Attributes:
        text: 提取的文本内容
        metadata: 元数据字典，包含来源、页码、加载时间、错误信息等
    """
    text: str
    metadata: dict = field(default_factory=dict)
    
    def __repr__(self) -> str:
        preview = self.text[:80].replace('\n', ' ') + ('...' if len(self.text) > 80 else '')
        return f"Document(text='{preview}', metadata={self.metadata})"
    
    def __len__(self) -> int:
        return len(self.text)


class DocumentLoader(ABC):
    """
    文档加载器抽象基类。
    
    所有具体的文档加载器必须实现 load() 方法，
    接受文件路径，返回 Document 对象列表。
    """
    
    @abstractmethod
    def load(self, file_path: str) -> list[Document]:
        """
        加载文档并返回 Document 对象列表。
        
        Args:
            file_path: 文档文件路径
            
        Returns:
            Document 对象列表。如果加载失败，返回空列表（而非抛出异常）。
            错误信息记录在 Document.metadata 中。
        """
        pass
    
    def _validate_file(self, file_path: str) -> Optional[str]:
        """
        验证文件是否存在且可读。
        
        Returns:
            如果验证失败返回错误信息字符串；成功返回 None。
        """
        if not os.path.exists(file_path):
            return f"文件不存在: {file_path}"
        if not os.path.isfile(file_path):
            return f"路径不是文件: {file_path}"
        if os.path.getsize(file_path) == 0:
            return f"文件为空: {file_path}"
        if not os.access(file_path, os.R_OK):
            return f"文件不可读: {file_path}"
        return None


print("[OK] Document 和 DocumentLoader 已定义。")
print(f"Document 示例: {Document(text='Hello RAG', metadata={'source': 'test.txt', 'page': 1})}")

## 3. PDF加载器 - PyPDF2 vs pdfplumber

PDF是RAG系统中最常见也最具挑战性的文档格式。PDF的设计目标是 **呈现一致性**（无论在什么设备上看起来都一样），而非 **数据提取便利性**。因此，从PDF中提取文本是一个本质上不精确的过程。

### 两种主流策略

| 库 | 策略 | 优点 | 缺点 |
|-----|------|------|------|
| **PyPDF2** | 解析PDF内部对象 | 快速、轻量、纯Python | 文本顺序可能错乱，中文支持弱 |
| **pdfplumber** | 基于pdfminer，逐字符分析布局 | 文本顺序准确、支持表格提取 | 较慢、内存占用大 |

### 常见问题与处理

- **加密PDF** —— 尝试空密码或提示用户输入
- **图片PDF（扫描件）** —— 文本提取为空，需要OCR（见第7节）
- **损坏的PDF** —— 捕获异常，返回带有错误信息的Document
- **水印/页眉页脚** —— pdfplumber 可以区域过滤

下面我们分别实现两种加载器，然后进行质量对比。

In [ ]:
class PDFLoaderPyPDF2(DocumentLoader):
    """
    使用 PyPDF2 加载PDF文档。
    
    将PDF的每一页作为一个独立的 Document 对象返回。
    包含对加密文件、损坏文件、空文件的完整错误处理。
    """
    
    def __init__(self, extract_metadata: bool = True):
        """
        Args:
            extract_metadata: 是否提取PDF元信息（标题、作者等）
        """
        self.extract_metadata = extract_metadata
        if not PYPDF2_AVAILABLE:
            raise ImportError("PyPDF2 未安装。请运行: pip install PyPDF2")
    
    def load(self, file_path: str) -> list[Document]:
        """加载PDF并返回每页为一个Document。"""
        # 验证文件
        validation_error = self._validate_file(file_path)
        if validation_error:
            return [Document(text="", metadata={"error": validation_error, "source": file_path})]
        
        documents = []
        try:
            with open(file_path, 'rb') as f:
                reader = PdfReader(f)
                
                # 检查PDF是否加密
                if reader.is_encrypted:
                    try:
                        # 尝试空密码解密
                        result = reader.decrypt('')
                        if result == 0:
                            return [Document(
                                text="",
                                metadata={
                                    "error": "PDF已加密且无法自动解密",
                                    "source": file_path,
                                    "encrypted": True
                                }
                            )]
                    except Exception as decrypt_error:
                        return [Document(
                            text="",
                            metadata={
                                "error": f"PDF解密失败: {str(decrypt_error)}",
                                "source": file_path,
                                "encrypted": True
                            }
                        )]
                
                # 提取PDF元信息
                pdf_metadata = {}
                if self.extract_metadata and reader.metadata:
                    pdf_metadata = {
                        "title": reader.metadata.get("/Title", ""),
                        "author": reader.metadata.get("/Author", ""),
                        "subject": reader.metadata.get("/Subject", ""),
                        "creator": reader.metadata.get("/Creator", ""),
                        "producer": reader.metadata.get("/Producer", ""),
                    }
                
                total_pages = len(reader.pages)
                if total_pages == 0:
                    return [Document(
                        text="",
                        metadata={"error": "PDF不包含任何页面", "source": file_path}
                    )]
                
                # 逐页提取文本
                for page_num, page in enumerate(reader.pages, start=1):
                    try:
                        text = page.extract_text() or ""
                    except Exception as page_error:
                        text = ""
                        page_error_msg = str(page_error)
                    
                    metadata = {
                        "source": file_path,
                        "page": page_num,
                        "total_pages": total_pages,
                        "loader": "PyPDF2",
                        **pdf_metadata
                    }
                    
                    if 'page_error_msg' in locals():
                        metadata["error"] = f"第{page_num}页提取失败: {page_error_msg}"
                        del page_error_msg
                    
                    documents.append(Document(text=text, metadata=metadata))
        
        except Exception as e:
            documents = [Document(
                text="",
                metadata={"error": f"PDF加载失败: {str(e)}", "source": file_path}
            )]
        
        return documents
    
    @staticmethod
    def create_test_pdf(file_path: str, content_lines: list[str]) -> str:
        """
        创建用于测试的PDF文件。
        优先使用fpdf2库，如果不可用则写入带标记的文本文件。
        """
        if FPDF_AVAILABLE:
            pdf = FPDF()
            pdf.add_page()
            pdf.set_font("Helvetica", size=12)
            for line in content_lines:
                pdf.cell(200, 10, text=line, new_x="LMARGIN", new_y="NEXT")
            pdf.output(file_path)
        else:
            # 备用方案：创建最小有效PDF（手动构造）
            texts = "\n".join(content_lines)
            # 手动构造一个最小PDF
            pdf_content = PDFLoaderPyPDF2._build_minimal_pdf(texts)
            with open(file_path, 'wb') as f:
                f.write(pdf_content)
        
        return file_path
    
    @staticmethod
    def _build_minimal_pdf(text: str) -> bytes:
        """构建最小有效PDF的字节内容（无依赖后备方案）。"""
        # 转义特殊字符
        escaped = text.replace('\\', '\\\\').replace('(', '\\(').replace(')', '\\)')
        
        pdf = f"""%PDF-1.4
1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj
2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj
3 0 obj<</Type/Page/MediaBox[0 0 612 792]/Parent 2 0 R/Resources<<>>>>/Contents 4 0 R>>endobj
4 0 obj<</Length 44>>stream
BT /F1 12 Tf 72 720 Td ({escaped}) Tj ET
endstream
endobj
xref
0 5
0000000000 65535 f 
0000000009 00000 n 
0000000058 00000 n 
0000000115 00000 n 
0000000225 00000 n 
trailer<</Size 5/Root 1 0 R>>
startxref
317
%%EOF"""
        return pdf.encode('latin-1', errors='replace')


# 测试 PyPDF2 加载器
print("=== 测试 PDFLoaderPyPDF2 ===")
if PYPDF2_AVAILABLE:
    loader = PDFLoaderPyPDF2()
    
    # 创建测试PDF
    test_pdf_path = os.path.join(tempfile.gettempdir(), "test_pypdf2_demo.pdf")
    PDFLoaderPyPDF2.create_test_pdf(test_pdf_path, [
        "RAG Document Loading Test",
        "Page 1: Hello from PyPDF2!",
        "This is a test PDF document.",
        "It demonstrates document loading for RAG.",
    ])
    print(f"  测试PDF已创建: {test_pdf_path}")
    
    # 加载并显示结果
    docs = loader.load(test_pdf_path)
    for doc in docs:
        print(f"\n  --- {doc.metadata.get('page', '?')}/{doc.metadata.get('total_pages', '?')} ---")
        print(f"  文本: {doc.text[:200]}")
        print(f"  元数据: {doc.metadata}")
    
    # 测试错误处理：不存在的文件
    print("\n  --- 测试错误处理：不存在的文件 ---")
    error_docs = loader.load("/nonexistent/file.pdf")
    print(f"  错误文档: {error_docs[0].metadata}")
    
    # 清理
    if os.path.exists(test_pdf_path):
        os.remove(test_pdf_path)
else:
    print("  [SKIP] PyPDF2 未安装")

In [ ]:
class PDFLoaderPdfPlumber(DocumentLoader):
    """
    使用 pdfplumber 加载PDF文档。
    
    pdfplumber 基于 pdfminer.six，提供更精确的文本布局分析，
    尤其擅长处理多栏布局和表格。但速度较 PyPDF2 慢。
    """
    
    def __init__(self, extract_tables: bool = True, keep_blank_chars: bool = False):
        """
        Args:
            extract_tables: 是否尝试提取表格数据
            keep_blank_chars: 是否保留空白字符（影响文本布局）
        """
        self.extract_tables = extract_tables
        self.keep_blank_chars = keep_blank_chars
        if not PDFPLUMBER_AVAILABLE:
            raise ImportError("pdfplumber 未安装。请运行: pip install pdfplumber")
    
    def load(self, file_path: str) -> list[Document]:
        """加载PDF并返回每页为一个Document。"""
        validation_error = self._validate_file(file_path)
        if validation_error:
            return [Document(text="", metadata={"error": validation_error, "source": file_path})]
        
        documents = []
        try:
            with pdfplumber.open(file_path) as pdf:
                total_pages = len(pdf.pages)
                
                if total_pages == 0:
                    return [Document(
                        text="",
                        metadata={"error": "PDF不包含任何页面", "source": file_path}
                    )]
                
                # 提取PDF元信息
                pdf_metadata = {}
                if pdf.metadata:
                    pdf_metadata = {
                        "title": pdf.metadata.get("Title", ""),
                        "author": pdf.metadata.get("Author", ""),
                        "subject": pdf.metadata.get("Subject", ""),
                    }
                
                for page_num, page in enumerate(pdf.pages, start=1):
                    try:
                        # 提取文本
                        text = page.extract_text(x_tolerance=2, y_tolerance=2) or ""
                        
                        # 如果启用，尝试提取表格
                        table_text = ""
                        if self.extract_tables:
                            tables = page.extract_tables()
                            if tables:
                                table_text = "\n\n--- 表格数据 ---\n"
                                for t_idx, table in enumerate(tables):
                                    table_text += f"\n[表格 {t_idx + 1}]\n"
                                    for row in table:
                                        row_text = " | ".join(
                                            str(cell) if cell is not None else ""
                                            for cell in row
                                        )
                                        table_text += row_text + "\n"
                        
                        full_text = text + table_text
                        
                    except Exception as page_error:
                        full_text = ""
                        page_error_msg = str(page_error)
                    
                    metadata = {
                        "source": file_path,
                        "page": page_num,
                        "total_pages": total_pages,
                        "loader": "pdfplumber",
                        **pdf_metadata
                    }
                    
                    if 'page_error_msg' in locals():
                        metadata["error"] = f"第{page_num}页提取失败: {page_error_msg}"
                        del page_error_msg
                    
                    documents.append(Document(text=full_text, metadata=metadata))
        
        except Exception as e:
            documents = [Document(
                text="",
                metadata={"error": f"PDF加载失败: {str(e)}", "source": file_path}
            )]
        
        return documents


print("[OK] PDFLoaderPdfPlumber 已定义。")

In [ ]:
# === 对比两种PDF加载器的提取质量 ===

def create_sample_pdf_for_comparison(file_path: str):
    """
    创建一个包含多种内容的示例PDF，用于对比测试。
    包含：标题、正文段落、多行内容。
    """
    content = [
        "RAG Document Loading - Comparison Test",
        "",
        "Section 1: Introduction",
        "This document demonstrates the differences between PyPDF2 and pdfplumber",
        "text extraction quality. Both libraries are commonly used in RAG pipelines.",
        "",
        "Section 2: Technical Details",
        "PyPDF2 parses PDF internal objects directly and is fast.",
        "pdfplumber uses pdfminer for character-level layout analysis.",
        "",
        "Section 3: Key Considerations",
        "1. Text order accuracy matters for retrieval quality.",
        "2. Table extraction is a unique pdfplumber feature.",
        "3. Memory usage and speed trade-offs exist.",
    ]
    return PDFLoaderPyPDF2.create_test_pdf(file_path, content)


print("=" * 60)
print("PDF加载器对比测试")
print("=" * 60)

comparison_pdf = os.path.join(tempfile.gettempdir(), "comparison_test.pdf")
create_sample_pdf_for_comparison(comparison_pdf)
print(f"\n示例PDF已创建: {comparison_pdf}")
print(f"文件大小: {os.path.getsize(comparison_pdf)} bytes")

# PyPDF2 加载
if PYPDF2_AVAILABLE:
    print("\n" + "-" * 40)
    print("PyPDF2 提取结果:")
    print("-" * 40)
    pypdf2_loader = PDFLoaderPyPDF2()
    pypdf2_docs = pypdf2_loader.load(comparison_pdf)
    for doc in pypdf2_docs:
        print(f"\n[页 {doc.metadata['page']}] 字符数: {len(doc.text)}")
        print(f"内容预览:\n{doc.text[:500]}")

# pdfplumber 加载
if PDFPLUMBER_AVAILABLE:
    print("\n" + "-" * 40)
    print("pdfplumber 提取结果:")
    print("-" * 40)
    plumber_loader = PDFLoaderPdfPlumber()
    plumber_docs = plumber_loader.load(comparison_pdf)
    for doc in plumber_docs:
        print(f"\n[页 {doc.metadata['page']}] 字符数: {len(doc.text)}")
        print(f"内容预览:\n{doc.text[:500]}")

# 对比总结
print("\n" + "=" * 60)
print("对比总结:")
print("=" * 60)
if PYPDF2_AVAILABLE and PDFPLUMBER_AVAILABLE:
    pypdf2_len = sum(len(d.text) for d in pypdf2_docs)
    plumber_len = sum(len(d.text) for d in plumber_docs)
    print(f"  PyPDF2 总字符数:     {pypdf2_len}")
    print(f"  pdfplumber 总字符数: {plumber_len}")
    print(f"  差异: {abs(pypdf2_len - plumber_len)} 字符")
    print("\n  选择建议:")
    print("    - 快速处理、简单布局 -> PyPDF2")
    print("    - 需要表格提取、多栏布局 -> pdfplumber")
    print("    - 生产环境建议 -> 两者都试，取最佳结果")

# 清理
if os.path.exists(comparison_pdf):
    os.remove(comparison_pdf)
    print(f"\n[清理] 已删除测试文件: {comparison_pdf}")

## 4. Word文档加载器

.docx 文件是Office文档的标准格式，本质是一个ZIP压缩包，包含XML文件。python-docx 库可以解析这个结构并提取文本内容。

### 需要处理的内容类型

- **段落（Paragraphs）** —— 正文内容的主要载体
- **表格（Tables）** —— 结构化数据，需要保留行列关系
- **页眉/页脚（Headers/Footers）** —— 可能包含重要信息（章节名、页码等）
- **文本框（Text Boxes）** —— 特殊位置的内容

### 注意事项

- .doc（旧格式，二进制）不被python-docx支持，需要先转换为.docx
- 损坏的.docx（无效ZIP）需要捕获异常
- 表格中的合并单元格可能导致内容重复

In [ ]:
class DocxLoader(DocumentLoader):
    """
    使用 python-docx 加载 Word (.docx) 文档。
    
    提取段落文本、表格内容、页眉和页脚。
    每个 .docx 文件作为一个 Document 对象返回。
    """
    
    def __init__(self, include_headers_footers: bool = True, include_tables: bool = True):
        """
        Args:
            include_headers_footers: 是否提取页眉和页脚
            include_tables: 是否提取表格内容
        """
        self.include_headers_footers = include_headers_footers
        self.include_tables = include_tables
        if not DOCX_AVAILABLE:
            raise ImportError("python-docx 未安装。请运行: pip install python-docx")
    
    def load(self, file_path: str) -> list[Document]:
        """加载.docx文件并提取文本内容。"""
        validation_error = self._validate_file(file_path)
        if validation_error:
            return [Document(text="", metadata={"error": validation_error, "source": file_path})]
        
        # 验证文件扩展名
        if not file_path.lower().endswith('.docx'):
            return [Document(
                text="",
                metadata={"error": "文件不是.docx格式", "source": file_path}
            )]
        
        try:
            doc = DocxDocument(file_path)
            text_parts = []
            
            # 提取文档属性
            props = doc.core_properties
            doc_metadata = {
                "title": props.title or "",
                "author": props.author or "",
                "created": str(props.created) if props.created else "",
                "modified": str(props.modified) if props.modified else "",
                "last_modified_by": props.last_modified_by or "",
            }
            
            # 提取页眉
            if self.include_headers_footers:
                for section_idx, section in enumerate(doc.sections):
                    header = section.header
                    if header and header.paragraphs:
                        header_text = "\n".join(
                            p.text for p in header.paragraphs if p.text.strip()
                        )
                        if header_text.strip():
                            text_parts.append(f"[页眉-第{section_idx+1}节] {header_text}")
            
            # 提取段落
            paragraph_count = 0
            for para in doc.paragraphs:
                if para.text.strip():
                    # 检测段落样式以保留结构
                    style_name = para.style.name if para.style else "Normal"
                    if 'Heading' in style_name or 'heading' in style_name:
                        text_parts.append(f"\n## {para.text}")
                    else:
                        text_parts.append(para.text)
                    paragraph_count += 1
            
            # 提取表格
            if self.include_tables:
                for t_idx, table in enumerate(doc.tables):
                    table_text = f"\n[表格 {t_idx + 1}]\n"
                    for r_idx, row in enumerate(table.rows):
                        cells = [cell.text.strip() for cell in row.cells]
                        table_text += " | ".join(cells) + "\n"
                        if r_idx == 0:  # 表头分隔线
                            table_text += "-" * (len(" | ".join(cells))) + "\n"
                    text_parts.append(table_text)
            
            # 提取页脚
            if self.include_headers_footers:
                for section_idx, section in enumerate(doc.sections):
                    footer = section.footer
                    if footer and footer.paragraphs:
                        footer_text = "\n".join(
                            p.text for p in footer.paragraphs if p.text.strip()
                        )
                        if footer_text.strip():
                            text_parts.append(f"[页脚-第{section_idx+1}节] {footer_text}")
            
            full_text = "\n".join(text_parts)
            
            if not full_text.strip():
                return [Document(
                    text="",
                    metadata={"warning": "文档不包含可提取的文本内容", "source": file_path, **doc_metadata}
                )]
            
            return [Document(
                text=full_text,
                metadata={
                    "source": file_path,
                    "loader": "DocxLoader",
                    "paragraphs": paragraph_count,
                    "tables": len(doc.tables),
                    **doc_metadata
                }
            )]
        
        except Exception as e:
            error_msg = str(e)
            if "not a valid" in error_msg.lower() or "zip" in error_msg.lower():
                error_msg = f"文件损坏或不是有效的.docx文件: {error_msg}"
            return [Document(
                text="",
                metadata={"error": error_msg, "source": file_path}
            )]


# 测试 DocxLoader
print("=== 测试 DocxLoader ===")
if DOCX_AVAILABLE:
    import tempfile
    
    # 创建测试 .docx 文件
    test_docx_path = os.path.join(tempfile.gettempdir(), "test_rag_demo.docx")
    
    test_doc = DocxDocument()
    test_doc.core_properties.title = "RAG Document Loading Guide"
    test_doc.core_properties.author = "RAG Team"
    
    test_doc.add_heading('Document Loading for RAG Systems', level=1)
    test_doc.add_paragraph(
        'Document loading is the first step in any RAG pipeline. '
        'It converts raw files into structured text that can be indexed and retrieved.'
    )
    test_doc.add_heading('Key Principles', level=2)
    test_doc.add_paragraph(
        '1. Unified interface: All loaders follow the same contract.\n'
        '2. Graceful degradation: Never crash on a bad file.\n'
        '3. Metadata preservation: Track source, page, and other context.'
    )
    
    # 添加表格
    table = test_doc.add_table(rows=3, cols=2, style='Light Shading Accent 1')
    table.cell(0, 0).text = 'Format'
    table.cell(0, 1).text = 'Loader'
    table.cell(1, 0).text = 'PDF'
    table.cell(1, 1).text = 'PyPDF2 / pdfplumber'
    table.cell(2, 0).text = 'Word'
    table.cell(2, 1).text = 'python-docx'
    
    test_doc.save(test_docx_path)
    print(f"  测试文件已创建: {test_docx_path}")
    print(f"  文件大小: {os.path.getsize(test_docx_path)} bytes")
    
    # 加载并显示
    docx_loader = DocxLoader()
    docs = docx_loader.load(test_docx_path)
    for doc in docs:
        print(f"\n  文本长度: {len(doc.text)} 字符")
        print(f"  元数据: {json.dumps(doc.metadata, indent=2, ensure_ascii=False)}")
        print(f"  内容预览:\n{doc.text[:600]}")
    
    # 测试损坏文件
    print("\n  --- 测试损坏的.docx文件 ---")
    corrupt_path = os.path.join(tempfile.gettempdir(), "corrupt.docx")
    with open(corrupt_path, 'w') as f:
        f.write("This is not a valid docx file.")
    error_docs = docx_loader.load(corrupt_path)
    print(f"  错误信息: {error_docs[0].metadata.get('error', 'N/A')}")
    os.remove(corrupt_path)
    
    # 清理
    if os.path.exists(test_docx_path):
        os.remove(test_docx_path)
else:
    print("  [SKIP] python-docx 未安装")

## 5. Markdown加载器

Markdown 是技术文档和知识库中最常见的格式之一。对于RAG系统，我们需要将Markdown的格式化标记转换为干净的纯文本，同时尽可能保留语义结构。

### 处理策略

我们使用 Python 的 `markdown` 库将 Markdown 转换为 HTML，然后用 BeautifulSoup 清洗HTML标签。这种"Markdown -> HTML -> 纯文本"的管线虽然迂回，但能正确处理：

- **代码块** —— 保留代码内容，去除```标记
- **链接** —— `[text](url)` 保留text，可选择保留url
- **图片** —— `![alt](url)` 保留alt文本或url
- **表格** —— 转换为可读的文本格式
- **列表** —— 保留缩进和层级结构

In [ ]:
class MarkdownLoader(DocumentLoader):
    """
    加载 Markdown (.md) 文件并转换为纯文本。
    
    使用 markdown 库将 .md 转换为 HTML，然后用 BeautifulSoup 清洗为纯文本。
    这样可以统一处理各种 Markdown 元素。
    """
    
    def __init__(self, keep_urls: bool = False, keep_images: bool = True):
        """
        Args:
            keep_urls: 是否在链接后保留URL（如 "text (url)"）
            keep_images: 是否保留图片的alt文本
        """
        self.keep_urls = keep_urls
        self.keep_images = keep_images
        if not MARKDOWN_AVAILABLE:
            raise ImportError("markdown 库未安装。请运行: pip install markdown")
        if not BS4_AVAILABLE:
            raise ImportError("beautifulsoup4 未安装。请运行: pip install beautifulsoup4")
    
    def load(self, file_path: str) -> list[Document]:
        """加载Markdown文件。"""
        validation_error = self._validate_file(file_path)
        if validation_error:
            return [Document(text="", metadata={"error": validation_error, "source": file_path})]
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_md = f.read()
        except UnicodeDecodeError:
            # 尝试其他编码
            try:
                with open(file_path, 'r', encoding='gbk') as f:
                    raw_md = f.read()
            except Exception as enc_error:
                return [Document(
                    text="",
                    metadata={"error": f"编码错误: {str(enc_error)}", "source": file_path}
                )]
        
        try:
            # Markdown -> HTML
            md_processor = markdown.Markdown(extensions=['extra', 'codehilite', 'tables'])
            html_content = md_processor.convert(raw_md)
            
            # HTML -> 纯文本
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # 移除不需要的标签
            for tag in soup(['script', 'style', 'meta', 'noscript']):
                tag.decompose()
            
            # 处理链接：保留文本，可选保留URL
            if self.keep_urls:
                for a_tag in soup.find_all('a', href=True):
                    a_tag.string = f"{a_tag.get_text()} ({a_tag['href']})"
            
            # 处理图片：保留alt文本
            if self.keep_images:
                for img_tag in soup.find_all('img', alt=True):
                    img_tag.string = f"[图片: {img_tag['alt']}]"
            else:
                for img_tag in soup.find_all('img'):
                    img_tag.decompose()
            
            # 提取文本
            text = soup.get_text(separator='\n', strip=True)
            
            # 清理多余空行
            lines = [line.strip() for line in text.split('\n')]
            cleaned_lines = []
            prev_empty = False
            for line in lines:
                if line == "":
                    if not prev_empty:
                        cleaned_lines.append(line)
                    prev_empty = True
                else:
                    cleaned_lines.append(line)
                    prev_empty = False
            text = "\n".join(cleaned_lines).strip()
            
            return [Document(
                text=text,
                metadata={
                    "source": file_path,
                    "loader": "MarkdownLoader",
                    "raw_length": len(raw_md),
                    "cleaned_length": len(text),
                }
            )]
        
        except Exception as e:
            return [Document(
                text="",
                metadata={"error": f"Markdown解析失败: {str(e)}", "source": file_path}
            )]


# 测试 MarkdownLoader
print("=== 测试 MarkdownLoader ===")
if MARKDOWN_AVAILABLE and BS4_AVAILABLE:
    sample_md = """# RAG Document Loading

## Introduction

This is a **sample** Markdown document for testing.

### Key Features

- Feature 1: PDF Support
- Feature 2: Word Support  
- Feature 3: HTML Support

### Code Example

```python
def load_document(path: str) -> list[Document]:
    return loader.load(path)
```

### Links and Images

See the [RAG documentation](https://example.com/rag) for details.

![RAG Architecture](https://example.com/rag.png)

### Table

| Format | Library | Status |
|--------|---------|--------|
| PDF    | PyPDF2  | Ready  |
| Word   | docx    | Ready  |
| HTML   | BS4     | Ready  |

> **Note:** This is a blockquote with important information.
"""
    
    # 写入临时文件
    test_md_path = os.path.join(tempfile.gettempdir(), "test_rag_demo.md")
    with open(test_md_path, 'w', encoding='utf-8') as f:
        f.write(sample_md)
    
    # 加载并显示结果
    md_loader = MarkdownLoader(keep_urls=True, keep_images=True)
    docs = md_loader.load(test_md_path)
    
    for doc in docs:
        print(f"\n  原始长度: {doc.metadata['raw_length']} 字符")
        print(f"  清洗后长度: {doc.metadata['cleaned_length']} 字符")
        print(f"  压缩率: {doc.metadata['cleaned_length'] / doc.metadata['raw_length'] * 100:.1f}%")
        print(f"  清洗后文本:\n{doc.text}")
    
    # 清理
    if os.path.exists(test_md_path):
        os.remove(test_md_path)
else:
    if not MARKDOWN_AVAILABLE:
        print("  [SKIP] markdown 库未安装")
    if not BS4_AVAILABLE:
        print("  [SKIP] beautifulsoup4 未安装")

## 6. HTML加载器

网页和HTML文档是RAG知识库的另一个重要来源。原始HTML中包含大量对检索无用的内容：

- `<script>` 和 `<style>` 标签 —— JavaScript代码和CSS样式
- 导航栏、侧边栏、页脚 —— 模板噪音
- HTML注释、meta标签、隐藏元素

### 清洗策略

BeautifulSoup 提供了强大的HTML解析和清洗能力：

1. **移除噪音标签** —— script, style, nav, footer, iframe, noscript, meta
2. **提取主要内容** —— 优先从 `<main>`, `<article>` 等语义标签中提取
3. **保留结构** —— 标题层级(h1-h6)、段落、列表
4. **编码处理** —— 自动检测或手动指定编码

In [ ]:
class HTMLLoader(DocumentLoader):
    """
    使用 BeautifulSoup 加载和清洗 HTML 文件。
    
    移除脚本、样式、导航等噪音元素，提取有意义的文本内容。
    """
    
    # 默认要移除的标签
    DEFAULT_REMOVE_TAGS = [
        'script', 'style', 'nav', 'footer', 'iframe',
        'noscript', 'meta', 'link', 'head', 'header'
    ]
    
    def __init__(
        self,
        remove_tags: list[str] = None,
        prefer_semantic: bool = True,
        encoding: str = None
    ):
        """
        Args:
            remove_tags: 要移除的HTML标签列表
            prefer_semantic: 是否优先提取语义标签内容（<main>, <article>）
            encoding: 文件编码，None则自动检测
        """
        self.remove_tags = remove_tags or self.DEFAULT_REMOVE_TAGS
        self.prefer_semantic = prefer_semantic
        self.encoding = encoding
        if not BS4_AVAILABLE:
            raise ImportError("beautifulsoup4 未安装。请运行: pip install beautifulsoup4")
    
    def load(self, file_path: str) -> list[Document]:
        """加载并清洗HTML文件。"""
        validation_error = self._validate_file(file_path)
        if validation_error:
            return [Document(text="", metadata={"error": validation_error, "source": file_path})]
        
        # 读取HTML文件（处理编码）
        raw_html = self._read_html(file_path)
        if raw_html is None:
            return [Document(
                text="",
                metadata={"error": "无法读取HTML文件", "source": file_path}
            )]
        
        try:
            soup = BeautifulSoup(raw_html, 'html.parser')
            
            # 记录清洗前的状态
            original_text_len = len(soup.get_text())
            
            # 移除不需要的标签
            for tag_name in self.remove_tags:
                for tag in soup.find_all(tag_name):
                    tag.decompose()
            
            # 移除HTML注释
            from bs4 import Comment
            for comment in soup.find_all(string=lambda text: isinstance(text, Comment)):
                comment.extract()
            
            # 如果启用语义提取，优先从语义标签中提取
            if self.prefer_semantic:
                main_content = soup.find('main') or soup.find('article') or soup.find(
                    'div', class_='content'
                ) or soup.find('div', id='content')
                if main_content:
                    soup = main_content
            else:
                # 取body内容
                body = soup.find('body')
                if body:
                    soup = body
            
            # 提取文本
            text = soup.get_text(separator='\n', strip=True)
            
            # 清理多余空行
            lines = [line.strip() for line in text.split('\n')]
            cleaned_lines = []
            prev_empty = False
            for line in lines:
                if line == "":
                    if not prev_empty:
                        cleaned_lines.append(line)
                    prev_empty = True
                else:
                    cleaned_lines.append(line)
                    prev_empty = False
            text = "\n".join(cleaned_lines).strip()
            
            # 提取标题
            title_tag = soup.find('title')
            if title_tag is None:
                # 如果已经移除了head中的title，从原始soup中找
                original_soup = BeautifulSoup(raw_html, 'html.parser')
                title_tag = original_soup.find('title')
            page_title = title_tag.get_text(strip=True) if title_tag else ""
            
            return [Document(
                text=text,
                metadata={
                    "source": file_path,
                    "loader": "HTMLLoader",
                    "title": page_title,
                    "original_text_length": original_text_len,
                    "cleaned_text_length": len(text),
                    "noise_removed": original_text_len - len(text),
                }
            )]
        
        except Exception as e:
            return [Document(
                text="",
                metadata={"error": f"HTML解析失败: {str(e)}", "source": file_path}
            )]
    
    def _read_html(self, file_path: str) -> Optional[str]:
        """读取HTML文件，自动处理编码。"""
        if self.encoding:
            encodings_to_try = [self.encoding]
        else:
            encodings_to_try = ['utf-8', 'gbk', 'gb2312', 'latin-1', 'iso-8859-1', 'cp1252']
        
        for enc in encodings_to_try:
            try:
                with open(file_path, 'r', encoding=enc) as f:
                    return f.read()
            except (UnicodeDecodeError, UnicodeError):
                continue
            except Exception:
                continue
        
        # 最后尝试：二进制读取 + chardet（如果可用）
        try:
            with open(file_path, 'rb') as f:
                raw_bytes = f.read()
            # 简单启发式：检查BOM
            if raw_bytes.startswith(b'\xef\xbb\xbf'):
                return raw_bytes.decode('utf-8-sig')
            return raw_bytes.decode('utf-8', errors='replace')
        except Exception:
            return None


# 测试 HTMLLoader
print("=== 测试 HTMLLoader ===")
if BS4_AVAILABLE:
    sample_html = """<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <title>RAG Document Loading Guide</title>
    <style>body { font-family: sans-serif; }</style>
    <script>console.log('tracking...');</script>
</head>
<body>
    <nav>
        <a href="/">Home</a> | <a href="/docs">Docs</a>
    </nav>
    <header>
        <h1>RAG System Documentation</h1>
    </header>
    <main>
        <article>
            <h2>Document Loading</h2>
            <p>Document loading is the <strong>first step</strong> in any RAG pipeline.</p>
            <p>It converts raw files into structured text that can be indexed.</p>
            <h3>Supported Formats</h3>
            <ul>
                <li>PDF (via PyPDF2 and pdfplumber)</li>
                <li>Word .docx (via python-docx)</li>
                <li>Markdown (via markdown library)</li>
                <li>HTML (via BeautifulSoup)</li>
            </ul>
            <p><!-- 这是注释，不应该出现在输出中 -->Key insight: clean extraction matters.</p>
        </article>
    </main>
    <footer>
        <p>&copy; 2024 RAG Team. All rights reserved.</p>
    </footer>
</body>
</html>"""
    
    # 写入临时文件
    test_html_path = os.path.join(tempfile.gettempdir(), "test_rag_demo.html")
    with open(test_html_path, 'w', encoding='utf-8') as f:
        f.write(sample_html)
    
    # 加载并清洗
    html_loader = HTMLLoader(prefer_semantic=True)
    docs = html_loader.load(test_html_path)
    
    for doc in docs:
        print(f"\n  页面标题: {doc.metadata.get('title', 'N/A')}")
        print(f"  清洗前文本长度: {doc.metadata['original_text_length']}")
        print(f"  清洗后文本长度: {doc.metadata['cleaned_text_length']}")
        print(f"  移除噪音: {doc.metadata['noise_removed']} 字符")
        print(f"\n  清洗后内容:\n{doc.text}")
    
    # 对比：不清洗的结果
    print("\n  --- 对比：如果不进行清洗 ---")
    raw_loader = HTMLLoader(remove_tags=[], prefer_semantic=False)
    raw_docs = raw_loader.load(test_html_path)
    if raw_docs:
        print(f"  未清洗文本长度: {len(raw_docs[0].text)} 字符")
        print(f"  清洗可移除: {len(raw_docs[0].text) - len(docs[0].text)} 字符的噪音")
    
    # 清理
    if os.path.exists(test_html_path):
        os.remove(test_html_path)
else:
    print("  [SKIP] beautifulsoup4 未安装")

## 7. OCR加载器（扫描文档）

许多现实场景中的文档是 **扫描件**——即包含文字的图片，而非可选择的文本。例如：

- 合同扫描件
- 书籍/论文的扫描版PDF
- 收据、发票的照片
- 手写笔记

对于这类文档，我们需要 **OCR（光学字符识别）**。

### 技术栈

- **Tesseract OCR** —— Google维护的开源OCR引擎，支持100+语言
- **Pillow (PIL)** —— Python图像处理库，用于预处理图像
- **pytesseract** —— Tesseract的Python封装

### 安装Tesseract

pytesseract只是Tesseract的Python接口，你需要单独安装Tesseract引擎：

- **Windows**: 下载安装器 https://github.com/UB-Mannheim/tesseract/wiki
- **macOS**: `brew install tesseract`
- **Ubuntu**: `sudo apt install tesseract-ocr`

中文OCR需要额外下载中文语言包（chi_sim）。

In [ ]:
class OCRLoader(DocumentLoader):
    """
    使用 Tesseract OCR 从图像中提取文字。
    
    支持常见图像格式（PNG, JPG, JPEG, TIFF, BMP, GIF）。
    可配置语言、预处理选项。
    """
    
    SUPPORTED_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.tiff', '.tif', '.bmp', '.gif', '.webp'}
    
    def __init__(
        self,
        languages: str = 'chi_sim+eng',
        tesseract_cmd: str = None,
        preprocess: bool = True
    ):
        """
        Args:
            languages: OCR识别语言，如 'eng', 'chi_sim', 'chi_sim+eng'
            tesseract_cmd: tesseract可执行文件路径，None则使用默认路径
            preprocess: 是否对图像进行预处理（二值化、去噪）
        """
        self.languages = languages
        self.preprocess = preprocess
        
        if not TESSERACT_AVAILABLE:
            raise ImportError("pytesseract 未安装。请运行: pip install pytesseract")
        
        if tesseract_cmd:
            pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
    
    def load(self, file_path: str) -> list[Document]:
        """对图像文件执行OCR。"""
        validation_error = self._validate_file(file_path)
        if validation_error:
            return [Document(text="", metadata={"error": validation_error, "source": file_path})]
        
        ext = os.path.splitext(file_path)[1].lower()
        if ext not in self.SUPPORTED_EXTENSIONS:
            return [Document(
                text="",
                metadata={
                    "error": f"不支持的图像格式: {ext}，支持: {self.SUPPORTED_EXTENSIONS}",
                    "source": file_path
                }
            )]
        
        try:
            # 检查tesseract是否可用
            tesseract_version = pytesseract.get_tesseract_version()
        except Exception:
            return [Document(
                text="",
                metadata={
                    "error": (
                        "Tesseract OCR 引擎未找到。"
                        "请安装 Tesseract: https://github.com/UB-Mannheim/tesseract/wiki"
                    ),
                    "source": file_path
                }
            )]
        
        try:
            # 打开图像
            image = Image.open(file_path)
            original_size = image.size
            
            # 图像预处理
            if self.preprocess:
                image = self._preprocess_image(image)
            
            # 执行OCR
            ocr_config = f'-l {self.languages} --psm 3'
            text = pytesseract.image_to_string(image, config=ocr_config)
            
            # 获取更详细的信息（可选）
            try:
                ocr_data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
                confidence_scores = [
                    int(c) for c in ocr_data['conf'] if c != '-1'
                ]
                avg_confidence = (
                    sum(confidence_scores) / len(confidence_scores)
                    if confidence_scores else 0
                )
            except Exception:
                avg_confidence = 0
            
            return [Document(
                text=text.strip(),
                metadata={
                    "source": file_path,
                    "loader": "OCRLoader",
                    "languages": self.languages,
                    "image_size": f"{original_size[0]}x{original_size[1]}",
                    "ocr_confidence": round(avg_confidence, 1),
                    "tesseract_version": str(tesseract_version),
                }
            )]
        
        except Exception as e:
            return [Document(
                text="",
                metadata={"error": f"OCR处理失败: {str(e)}", "source": file_path}
            )]
    
    def _preprocess_image(self, image: Image.Image) -> Image.Image:
        """
        图像预处理以提高OCR准确率。
        
        步骤：
        1. 转换为灰度
        2. 二值化（Otsu阈值）
        3. 降噪（可选）
        """
        # 转换为灰度
        if image.mode != 'L':
            image = image.convert('L')
        
        # 二值化
        try:
            import numpy as np
            img_array = np.array(image)
            # 简单阈值
            threshold = 128
            binary = (img_array > threshold).astype(np.uint8) * 255
            image = Image.fromarray(binary, mode='L')
        except ImportError:
            # numpy不可用则跳过二值化
            pass
        
        return image


# 测试 OCRLoader
print("=== 测试 OCRLoader ===")
if TESSERACT_AVAILABLE:
    # 检查tesseract是否可用
    try:
        version = pytesseract.get_tesseract_version()
        print(f"  Tesseract 版本: {version}")
    except Exception:
        print("  [SKIP] Tesseract OCR 引擎未安装或不在PATH中")
        print("  请从 https://github.com/UB-Mannheim/tesseract/wiki 下载安装")
else:
    print("  [SKIP] pytesseract 未安装")

# 使用PIL创建带文字的测试图像
print("\n--- 创建测试图像 ---")
try:
    from PIL import Image, ImageDraw, ImageFont
    
    # 创建白色背景图像
    img = Image.new('RGB', (600, 200), color='white')
    draw = ImageDraw.Draw(img)
    
    # 尝试使用系统字体，失败则使用默认字体
    try:
        # Windows常见字体路径
        font_paths = [
            "C:/Windows/Fonts/arial.ttf",
            "C:/Windows/Fonts/Arial.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            "/Library/Fonts/Arial.ttf",
        ]
        font = None
        for fp in font_paths:
            if os.path.exists(fp):
                font = ImageFont.truetype(fp, 24)
                break
        if font is None:
            font = ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()
    
    # 绘制文字
    draw.text((40, 30), "RAG Document Loading Test", fill='black', font=font)
    draw.text((40, 70), "OCR enables text extraction from images.", fill='black', font=font)
    draw.text((40, 110), "This is a sample image with text content.", fill='black', font=font)
    draw.text((40, 150), "Tesseract is a powerful open-source OCR engine.", fill='black', font=font)
    
    test_img_path = os.path.join(tempfile.gettempdir(), "test_ocr_demo.png")
    img.save(test_img_path)
    print(f"  测试图像已创建: {test_img_path}")
    print(f"  图像大小: {os.path.getsize(test_img_path)} bytes")
    
    # 尝试OCR
    if TESSERACT_AVAILABLE:
        try:
            ocr_loader = OCRLoader(languages='eng')
            docs = ocr_loader.load(test_img_path)
            for doc in docs:
                if 'error' in doc.metadata:
                    print(f"\n  OCR错误: {doc.metadata['error']}")
                else:
                    print(f"\n  置信度: {doc.metadata.get('ocr_confidence', 'N/A')}%")
                    print(f"  识别文本:\n{doc.text}")
        except Exception as e:
            print(f"  OCR执行失败: {e}")
    
    # 清理
    if os.path.exists(test_img_path):
        os.remove(test_img_path)

except ImportError as e:
    print(f"  Pillow不可用: {e}")

## 8. 统一加载器工厂

现在我们有了多种文档加载器，每种针对不同格式。手动判断文件类型并选择正确的加载器会很繁琐。工厂模式（Factory Pattern）提供了一个优雅的解决方案：

### 工厂函数的设计目标

1. **自动检测文件类型** —— 基于扩展名选择加载器
2. **单一入口点** —— `load_document(file_path)` 处理所有格式
3. **可扩展** —— 轻松注册新的加载器
4. **清晰的错误信息** —— 对不支持的格式给出有意义的提示

In [ ]:
# === 加载器工厂：自动选择正确的加载器 ===

# 扩展名到加载器类的映射
LOADER_REGISTRY: dict[str, type] = {}


def register_loader(extensions: list[str]):
    """
    装饰器：将加载器类注册到指定的文件扩展名。
    
    Usage:
        @register_loader(['.pdf'])
        class PDFLoader(DocumentLoader):
            ...
    """
    def decorator(loader_cls: type):
        for ext in extensions:
            LOADER_REGISTRY[ext.lower()] = loader_cls
        return loader_cls
    return decorator


def load_document(
    file_path: str,
    loader_kwargs: dict = None
) -> list[Document]:
    """
    统一的文档加载入口。根据文件扩展名自动选择和实例化加载器。
    
    Args:
        file_path: 文档文件路径
        loader_kwargs: 传递给加载器构造函数的额外参数
        
    Returns:
        Document 对象列表
        
    Raises:
        ValueError: 不支持的文件类型
        
    Supported formats:
        .pdf  -> PDFLoaderPdfPlumber (preferred) / PDFLoaderPyPDF2 (fallback)
        .docx -> DocxLoader
        .md   -> MarkdownLoader
        .html, .htm -> HTMLLoader
        .png, .jpg, .jpeg, .tiff, .tif, .bmp, .gif -> OCRLoader
        .txt  -> Plain text (built-in)
    """
    if not os.path.exists(file_path):
        return [Document(
            text="",
            metadata={"error": f"文件不存在: {file_path}", "source": file_path}
        )]
    
    ext = os.path.splitext(file_path)[1].lower()
    loader_kwargs = loader_kwargs or {}
    
    # PDF: 优先使用 pdfplumber，回退到 PyPDF2
    if ext == '.pdf':
        if PDFPLUMBER_AVAILABLE:
            loader = PDFLoaderPdfPlumber(**loader_kwargs)
        elif PYPDF2_AVAILABLE:
            loader = PDFLoaderPyPDF2(**loader_kwargs)
        else:
            return [Document(
                text="",
                metadata={
                    "error": "未安装PDF加载库。请运行: pip install pdfplumber 或 pip install PyPDF2",
                    "source": file_path
                }
            )]
        return loader.load(file_path)
    
    # Word文档
    if ext == '.docx':
        if not DOCX_AVAILABLE:
            return [Document(
                text="",
                metadata={
                    "error": "python-docx 未安装。请运行: pip install python-docx",
                    "source": file_path
                }
            )]
        loader = DocxLoader(**loader_kwargs)
        return loader.load(file_path)
    
    # Markdown
    if ext == '.md':
        if not MARKDOWN_AVAILABLE or not BS4_AVAILABLE:
            missing = []
            if not MARKDOWN_AVAILABLE:
                missing.append("markdown")
            if not BS4_AVAILABLE:
                missing.append("beautifulsoup4")
            return [Document(
                text="",
                metadata={
                    "error": f"缺少依赖: {', '.join(missing)}。请运行 pip install",
                    "source": file_path
                }
            )]
        loader = MarkdownLoader(**loader_kwargs)
        return loader.load(file_path)
    
    # HTML
    if ext in ('.html', '.htm'):
        if not BS4_AVAILABLE:
            return [Document(
                text="",
                metadata={
                    "error": "beautifulsoup4 未安装。请运行: pip install beautifulsoup4",
                    "source": file_path
                }
            )]
        loader = HTMLLoader(**loader_kwargs)
        return loader.load(file_path)
    
    # 图像 (OCR)
    image_extensions = {'.png', '.jpg', '.jpeg', '.tiff', '.tif', '.bmp', '.gif', '.webp'}
    if ext in image_extensions:
        if not TESSERACT_AVAILABLE:
            return [Document(
                text="",
                metadata={
                    "error": "pytesseract 未安装。请运行: pip install pytesseract",
                    "source": file_path
                }
            )]
        loader = OCRLoader(**loader_kwargs)
        return loader.load(file_path)
    
    # 纯文本文件
    if ext == '.txt':
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
        except UnicodeDecodeError:
            try:
                with open(file_path, 'r', encoding='gbk') as f:
                    text = f.read()
            except Exception as e:
                return [Document(
                    text="",
                    metadata={"error": f"编码错误: {str(e)}", "source": file_path}
                )]
        except Exception as e:
            return [Document(
                text="",
                metadata={"error": f"读取文件失败: {str(e)}", "source": file_path}
            )]
        
        return [Document(
            text=text,
            metadata={
                "source": file_path,
                "loader": "TextLoader",
                "size_bytes": os.path.getsize(file_path),
                "lines": text.count('\n') + 1,
            }
        )]
    
    # 不支持的格式
    return [Document(
        text="",
        metadata={
            "error": f"不支持的文件类型: {ext}。支持的类型: .pdf, .docx, .md, .html, .htm, .txt, 和常见图像格式",
            "source": file_path
        }
    )]


# 测试工厂函数
print("=== 测试 load_document() 工厂函数 ===\n")

# 创建各种测试文件
test_dir = os.path.join(tempfile.gettempdir(), "rag_factory_test")
os.makedirs(test_dir, exist_ok=True)

# 创建测试 .txt 文件
txt_path = os.path.join(test_dir, "sample.txt")
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write("This is a plain text document.\nIt has multiple lines.\nUsed for testing.")

# 创建测试 .md 文件
md_path = os.path.join(test_dir, "sample.md")
with open(md_path, 'w', encoding='utf-8') as f:
    f.write("# Test Markdown\n\nThis is a **test** markdown file.\n\n- Item 1\n- Item 2")

# 创建测试 .html 文件
html_path = os.path.join(test_dir, "sample.html")
with open(html_path, 'w', encoding='utf-8') as f:
    f.write("<html><body><h1>Test HTML</h1><p>This is a paragraph.</p></body></html>")

# 测试每种格式
test_files = [
    ("文本文件", txt_path),
    ("Markdown文件", md_path),
    ("HTML文件", html_path),
    ("不存在的文件", os.path.join(test_dir, "nonexistent.pdf")),
    ("不支持的格式", os.path.join(test_dir, "data.xyz")),
]

for label, path in test_files:
    print(f"[{label}] {os.path.basename(path)}")
    docs = load_document(path)
    for doc in docs:
        if 'error' in doc.metadata:
            print(f"  ERROR: {doc.metadata['error']}")
        else:
            print(f"  OK: {len(doc.text)} 字符, loader={doc.metadata.get('loader', 'N/A')}")
            print(f"  预览: {doc.text[:100]}..." if len(doc.text) > 100 else f"  预览: {doc.text}")
    print()

# 清理
shutil.rmtree(test_dir, ignore_errors=True)
print("[清理] 测试文件已删除")

## 9. 批量加载目录

在实际RAG应用中，我们通常需要处理整个目录树中的文档——可能有成百上千个文件，格式各异。`load_directory()` 函数提供了批量加载能力。

### 设计要点

- **递归遍历** —— 使用 `os.walk()` 处理嵌套目录
- **错误聚合** —— 单个文件加载失败不中断整体流程
- **进度追踪** —— 实时显示加载进度
- **摘要报告** —— 统计成功/失败/跳过的文件
- **文件过滤** —— 支持扩展名白名单和黑名单

In [ ]:
def load_directory(
    directory: str,
    recursive: bool = True,
    include_extensions: list[str] = None,
    exclude_extensions: list[str] = None,
    show_progress: bool = True,
    max_files: int = None,
) -> tuple[list[Document], dict]:
    """
    递归加载目录中的所有支持文档。
    
    Args:
        directory: 目标目录路径
        recursive: 是否递归遍历子目录
        include_extensions: 仅加载指定扩展名的文件（白名单），None表示所有支持的类型
        exclude_extensions: 排除指定扩展名的文件（黑名单）
        show_progress: 是否打印进度信息
        max_files: 最多加载的文件数，None表示无限制
        
    Returns:
        (documents, report) 元组：
        - documents: Document对象列表
        - report: 统计摘要字典
    """
    # 默认支持的所有扩展名
    all_supported = {
        '.pdf', '.docx', '.md', '.html', '.htm', '.txt',
        '.png', '.jpg', '.jpeg', '.tiff', '.tif', '.bmp', '.gif', '.webp'
    }
    
    if include_extensions:
        target_extensions = {e.lower() if e.startswith('.') else f'.{e.lower()}' for e in include_extensions}
    else:
        target_extensions = all_supported
    
    if exclude_extensions:
        exclude_set = {e.lower() if e.startswith('.') else f'.{e.lower()}' for e in exclude_extensions}
        target_extensions = target_extensions - exclude_set
    
    # 收集所有目标文件
    file_paths = []
    if recursive:
        for root, dirs, files in os.walk(directory):
            for filename in files:
                ext = os.path.splitext(filename)[1].lower()
                if ext in target_extensions:
                    file_paths.append(os.path.join(root, filename))
    else:
        for filename in os.listdir(directory):
            full_path = os.path.join(directory, filename)
            if os.path.isfile(full_path):
                ext = os.path.splitext(filename)[1].lower()
                if ext in target_extensions:
                    file_paths.append(full_path)
    
    # 排序以保证可复现性
    file_paths.sort()
    
    if max_files:
        file_paths = file_paths[:max_files]
    
    total_files = len(file_paths)
    
    if show_progress:
        print(f"[批量加载] 目录: {directory}")
        print(f"[批量加载] 发现 {total_files} 个文件")
        print(f"[批量加载] 目标扩展名: {sorted(target_extensions)}")
        print()
    
    # 逐文件加载
    all_documents = []
    stats = {
        "total_found": total_files,
        "success": 0,
        "failed": 0,
        "empty": 0,
        "skipped": 0,
        "total_text_length": 0,
        "errors": [],
        "files_by_type": {},
    }
    
    for idx, file_path in enumerate(file_paths, start=1):
        ext = os.path.splitext(file_path)[1].lower()
        
        if show_progress:
            print(f"  [{idx}/{total_files}] {os.path.basename(file_path)}", end=" ... ")
        
        try:
            docs = load_document(file_path)
            
            # 检查是否有错误
            has_error = any('error' in doc.metadata for doc in docs)
            
            if has_error:
                stats["failed"] += 1
                error_msgs = [doc.metadata['error'] for doc in docs if 'error' in doc.metadata]
                stats["errors"].append({
                    "file": file_path,
                    "errors": error_msgs
                })
                if show_progress:
                    print(f"FAILED: {'; '.join(error_msgs)}")
                continue
            
            # 检查是否有空文档
            total_len = sum(len(doc.text) for doc in docs)
            if total_len == 0:
                stats["empty"] += 1
                if show_progress:
                    print("EMPTY (无可提取文本)")
                continue
            
            # 成功
            stats["success"] += 1
            stats["total_text_length"] += total_len
            stats["files_by_type"][ext] = stats["files_by_type"].get(ext, 0) + 1
            all_documents.extend(docs)
            
            if show_progress:
                print(f"OK ({total_len} 字符, {len(docs)} 个文档片段)")
        
        except Exception as e:
            stats["failed"] += 1
            stats["errors"].append({
                "file": file_path,
                "errors": [str(e)]
            })
            if show_progress:
                print(f"EXCEPTION: {str(e)}")
    
    # 打印摘要
    if show_progress:
        print()
        print("=" * 50)
        print("批量加载摘要")
        print("=" * 50)
        print(f"  发现文件:     {stats['total_found']}")
        print(f"  成功加载:     {stats['success']}")
        print(f"  加载失败:     {stats['failed']}")
        print(f"  空文档:       {stats['empty']}")
        print(f"  总文档片段:   {len(all_documents)}")
        print(f"  总字符数:     {stats['total_text_length']}")
        if stats['files_by_type']:
            print(f"  文件类型分布:")
            for ext, count in sorted(stats['files_by_type'].items()):
                print(f"    {ext}: {count} 个文件")
        if stats['errors']:
            print(f"\n  错误详情 (前5条):")
            for err in stats['errors'][:5]:
                print(f"    - {os.path.basename(err['file'])}: {'; '.join(err['errors'])}")
    
    return all_documents, stats


# 简单验证
print("[OK] load_directory() 函数已定义。\n")
print("函数签名:")
print("  load_directory(directory, recursive=True, include_extensions=None, exclude_extensions=None, show_progress=True, max_files=None)")
print("  -> tuple[list[Document], dict]  # (documents, report)")

## 10. 练习：加载混合格式目录

现在运用你所学的知识，完成以下练习。

### 练习目标

创建一个包含多种格式文件的临时目录，使用 `load_directory()` 批量加载，验证结果。

### 具体要求

1. 创建一个临时目录，包含至少以下文件：
   - 1个 .txt 文件（包含英文段落）
   - 1个 .md 文件（包含标题、列表、代码块）
   - 1个 .html 文件（包含导航、正文、脚本）
   - 1个 .docx 文件（包含标题、段落、表格）
   - 1个 .pdf 文件（如果可用）
2. 调用 `load_directory()` 加载整个目录
3. 验证返回的文档数量和统计信息
4. 打印每个文档的来源和文本长度

### 提示

- 使用 `tempfile.mkdtemp()` 创建临时目录
- 使用前面定义的辅助方法创建测试文件
- 思考：如果某个格式的库未安装，`load_document()` 会如何处理？

In [ ]:
# === 练习：加载混合格式目录 ===

print("=" * 60)
print("练习：批量加载混合格式目录")
print("=" * 60)

# 步骤1：创建临时目录
exercise_dir = tempfile.mkdtemp(prefix="rag_exercise_")
print(f"\n临时目录: {exercise_dir}")

# 步骤2：创建各种格式的测试文件

# 2a. 创建 .txt 文件
txt_content = """Document Loading in RAG Systems

Document loading is the foundational step in any Retrieval-Augmented Generation pipeline.
It converts heterogeneous file formats into a unified text representation that can be
indexed, chunked, and retrieved during query time.

Key challenges include:
- Handling diverse file formats (PDF, Word, HTML, Markdown, images)
- Graceful error handling for corrupted or encrypted files
- Preserving document structure and metadata
- Scaling to large document collections efficiently
"""
txt_file = os.path.join(exercise_dir, "01_introduction.txt")
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write(txt_content)
print(f"  [创建] {txt_file}")

# 2b. 创建 .md 文件
md_content = """# RAG Pipeline Overview

## Phase 1: Document Loading

The first step converts raw documents into structured text.

### Supported Formats

- **PDF**: Using PyPDF2 and pdfplumber
- **Word**: Using python-docx
- **Markdown**: Using markdown library
- **HTML**: Using BeautifulSoup

### Code Example

```python
from rag.loaders import load_document

docs = load_document("report.pdf")
for doc in docs:
    print(f"Page {doc.metadata['page']}: {len(doc.text)} chars")
```

## Phase 2: Text Splitting

After loading, documents are split into manageable chunks.

> **Note**: Chunk size and overlap are critical parameters.
"""
md_file = os.path.join(exercise_dir, "02_pipeline_overview.md")
with open(md_file, 'w', encoding='utf-8') as f:
    f.write(md_content)
print(f"  [创建] {md_file}")

# 2c. 创建 .html 文件
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>RAG Best Practices</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; }
        .nav { background: #f0f0f0; padding: 10px; }
        .footer { color: #666; font-size: 12px; }
    </style>
    <script>
        console.log('Page loaded');
        // Tracking code - should be removed during loading
    </script>
</head>
<body>
    <nav class="nav">
        <a href="/">Home</a> |
        <a href="/docs">Documentation</a> |
        <a href="/api">API</a>
    </nav>
    <main>
        <article>
            <h1>RAG Best Practices for Document Loading</h1>
            <section>
                <h2>1. Always Validate Input</h2>
                <p>Check file existence, size, and readability before attempting extraction.</p>
            </section>
            <section>
                <h2>2. Preserve Metadata</h2>
                <p>Track the source file path, page number, section, and other context information
                to enable citation and debugging in the final RAG output.</p>
            </section>
            <section>
                <h2>3. Handle Errors Gracefully</h2>
                <p>Never let a single corrupted file crash your entire loading pipeline.
                Log errors and continue processing the remaining documents.</p>
            </section>
        </article>
    </main>
    <footer class="footer">
        <p>RAG Loading Framework v1.0 | Documentation</p>
    </footer>
</body>
</html>"""
html_file = os.path.join(exercise_dir, "03_best_practices.html")
with open(html_file, 'w', encoding='utf-8') as f:
    f.write(html_content)
print(f"  [创建] {html_file}")

# 2d. 创建 .docx 文件（如果可用）
if DOCX_AVAILABLE:
    docx_file = os.path.join(exercise_dir, "04_technical_specs.docx")
    doc = DocxDocument()
    doc.core_properties.title = "RAG Technical Specifications"
    doc.core_properties.author = "Engineering Team"
    
    doc.add_heading('Technical Specifications', level=1)
    doc.add_paragraph(
        'This document outlines the technical requirements for the RAG document loading system.'
    )
    
    doc.add_heading('Performance Requirements', level=2)
    doc.add_paragraph(
        'The loading pipeline must handle 10,000+ documents per batch with sub-second per-document latency.'
    )
    
    doc.add_heading('Supported Formats', level=2)
    
    table = doc.add_table(rows=4, cols=3, style='Light Shading Accent 1')
    table.cell(0, 0).text = 'Format'
    table.cell(0, 1).text = 'Library'
    table.cell(0, 2).text = 'Status'
    table.cell(1, 0).text = 'PDF'
    table.cell(1, 1).text = 'pdfplumber'
    table.cell(1, 2).text = 'Production'
    table.cell(2, 0).text = 'Word'
    table.cell(2, 1).text = 'python-docx'
    table.cell(2, 2).text = 'Production'
    table.cell(3, 0).text = 'OCR'
    table.cell(3, 1).text = 'Tesseract'
    table.cell(3, 2).text = 'Beta'
    
    doc.save(docx_file)
    print(f"  [创建] {docx_file}")
else:
    docx_file = None
    print("  [跳过] python-docx 未安装，跳过.docx文件创建")

# 2e. 创建 .pdf 文件（如果可用）
pdf_file = None
if PYPDF2_AVAILABLE and FPDF_AVAILABLE:
    pdf_file = os.path.join(exercise_dir, "05_executive_summary.pdf")
    PDFLoaderPyPDF2.create_test_pdf(pdf_file, [
        "Executive Summary: RAG Document Loading",
        "",
        "This document summarizes the document loading capabilities",
        "of our RAG system. The system supports multiple formats",
        "including PDF, Word, HTML, Markdown, and images via OCR.",
        "",
        "Key metrics:",
        "- 6 document types supported",
        "- 99.5% successful extraction rate",
        "- Average 0.3s per document",
    ])
    print(f"  [创建] {pdf_file}")
elif not PYPDF2_AVAILABLE:
    print("  [跳过] PyPDF2 未安装，跳过.pdf文件创建")
elif not FPDF_AVAILABLE:
    # 使用后备方法创建PDF
    pdf_file = os.path.join(exercise_dir, "05_executive_summary.pdf")
    PDFLoaderPyPDF2.create_test_pdf(pdf_file, [
        "Executive Summary: RAG Document Loading",
        "This is a minimal test PDF.",
    ])
    print(f"  [创建] {pdf_file} (最小PDF，fpdf2不可用)")

# 步骤3：批量加载
print(f"\n{'='*60}")
print("开始批量加载...")
print(f"{'='*60}")

documents, report = load_directory(
    exercise_dir,
    recursive=False,
    show_progress=True,
)

# 步骤4：验证结果
print(f"\n{'='*60}")
print("验证结果")
print(f"{'='*60}")
print(f"  期望文件数: ~5 (取决于可用的库)")
print(f"  成功加载:   {report['success']}")
print(f"  加载失败:   {report['failed']}")
print(f"  空文档:     {report['empty']}")
print(f"  文档片段:   {len(documents)}")
print(f"  总字符数:   {report['total_text_length']}")

print(f"\n--- 文档详情 ---")
for i, doc in enumerate(documents, start=1):
    source = doc.metadata.get('source', '?')
    loader = doc.metadata.get('loader', '?')
    page = doc.metadata.get('page', None)
    page_info = f", page={page}" if page else ""
    print(f"\n  [{i}] {os.path.basename(source)}")
    print(f"      Loader: {loader}{page_info}")
    print(f"      Length: {len(doc.text)} chars")
    print(f"      Preview: {doc.text[:120].replace(chr(10), ' ')}...")

# 步骤5：测试错误处理 —— 创建一个损坏的文件并验证不影响其他文件
print(f"\n{'='*60}")
print("测试错误处理：添加一个不存在的格式")
print(f"{'='*60}")
unknown_file = os.path.join(exercise_dir, "06_unknown.xyz")
with open(unknown_file, 'w') as f:
    f.write("This format is not supported.")
print(f"  添加了不支持格式的文件: {unknown_file}")

documents2, report2 = load_directory(exercise_dir, recursive=False, show_progress=True)
print(f"\n  结果: 仍然成功加载 {report2['success']} 个文件，不支持的文件被优雅跳过")

# 清理
print(f"\n[清理] 删除临时目录: {exercise_dir}")
shutil.rmtree(exercise_dir, ignore_errors=True)

print(f"\n[OK] 练习完成！")

## 练习解答与提示

### 预期结果

- **成功加载** 4-5个文件（.txt, .md, .html 必定成功；.docx 和 .pdf 取决于库是否安装）
- **加载失败** 1个（.xyz 不被支持）
- **文档片段** 因PDF按页分割，所以可能多于文件数

### 常见问题与解决方案

| 问题 | 原因 | 解决 |
|------|------|------|
| `ImportError` | 缺少依赖库 | 运行对应的 `pip install` 命令 |
| OCR返回乱码 | 语言包未安装 | 下载对应语言的traineddata |
| .docx加载失败 | 文件是旧.doc格式 | 用Word转换为.docx |
| PDF中文乱码 | PDF字体未嵌入 | 尝试使用pdfplumber替代PyPDF2 |
| 大文件内存溢出 | 整个文件读入内存 | 实现流式分页加载 |

### 进阶挑战

1. **添加新格式支持** —— 实现 `PPTXLoader`（使用 python-pptx 库）
2. **并行加载** —— 使用 `concurrent.futures` 加速批量加载
3. **流式处理** —— 对大文件实现生成器模式，避免一次性加载全部内容
4. **内容去重** —— 在批量加载时检测并跳过重复文档
5. **增量加载** —— 仅加载自上次运行以来修改过的文件

## 11. 总结

### 关键要点

1. **统一接口是基石** —— `Document` + `DocumentLoader` 抽象为系统提供了可扩展的架构基础。所有加载器遵循相同的契约，新格式的添加不会影响现有代码。

2. **PDF加载需要策略选择** —— PyPDF2 快速轻量，pdfplumber 准确但慢。生产环境中建议同时使用两者并根据置信度选择最佳结果。

3. **HTML/Markdown需要清洗** —— 原始标记中包含大量对检索无用的噪音（脚本、样式、导航）。清洗步骤对RAG的检索质量至关重要。

4. **OCR是扫描文档的必经之路** —— Tesseract + 图像预处理能有效处理扫描件，但需要额外安装Tesseract引擎。

5. **错误处理不是可选项** —— 真实世界的文档千差万别，每个加载器都必须包含完整的错误处理。单点故障不应拖垮整个pipeline。

6. **工厂模式简化调用** —— `load_document()` 提供单点入口，根据扩展名自动分发，调用者无需关心具体加载器。

7. **批量加载需要统计** —— `load_directory()` 不仅加载文件，还提供进度追踪、错误聚合和摘要报告，使大规模文档处理可控、可观测。

### 本课构建的组件

```
Document (dataclass)          <- 统一数据模型
    |
DocumentLoader (ABC)          <- 抽象基类
    |
    +-- PDFLoaderPyPDF2        <- PyPDF2策略
    +-- PDFLoaderPdfPlumber    <- pdfplumber策略
    +-- DocxLoader             <- Word文档
    +-- MarkdownLoader         <- Markdown清洗
    +-- HTMLLoader             <- HTML清洗
    +-- OCRLoader              <- 图像OCR
    |
load_document()               <- 工厂函数（自动选择加载器）
    |
load_directory()              <- 批量加载（递归+统计）
```

### 下一课预告：文本分块（Text Splitting）

文档加载只是第一步。加载后的文档通常太长，无法直接输入到LLM的上下文窗口。下一课我们将学习如何将文档智能地分割成适合检索和生成的文本块（chunks），包括：

- **固定大小分块** —— 按字符数或token数分割
- **语义分块** —— 基于段落、句子边界分割
- **重叠分块** —— 保留上下文连贯性
- **递归分块** —— 使用分隔符优先级的多级分割

准备好进入RAG pipeline的下一阶段了吗？